# Shoe Purchase Problem in Pure Python
## Five Exact Methods for Book Problem 2.11


## Problem Statement

Zapato a tu medida wants to buy shoe boxes from two wholesalers, Nikis and Pinki, with a budget of
`50,000 USD`. The company wants the same number of boxes from each supplier, at least `250` pairs
of shoes in total, and the printed model in section `2.11.1` imposes a minimum purchase of `30`
boxes.

### Note on the book

The explanatory paragraph in section `2.11.2` says "como mínimo 50 cajas", but equation `(2.108)`
shows `sum x_i >= 30` and the published AMPL solution (`15` tennis boxes and `15` sandals boxes)
is only feasible with `30`. In these notebooks we follow the equations and the published solution.


In [ ]:
from __future__ import annotations

from functools import lru_cache, wraps
from time import perf_counter


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def choose_better(left, right):
    if left is None:
        return right
    if right is None:
        return left
    left_key = (left["total_utility"], -left["total_cost"], tuple(left["boxes"][name] for name in BOXES))
    right_key = (right["total_utility"], -right["total_cost"], tuple(right["boxes"][name] for name in BOXES))
    return right if right_key > left_key else left


BUDGET = 50000
MIN_TOTAL_BOXES = 30
MIN_SHOE_PAIRS = 250
BOXES = ["tennis", "boot", "heels", "sandals"]

COST = {"tennis": 2700, "boot": 2400, "heels": 640, "sandals": 625}
SHOE_PAIRS = {"tennis": 45, "boot": 30, "heels": 16, "sandals": 25}
UTILITY = {"tennis": 4500, "boot": 2550, "heels": 960, "sandals": 1250}

PAIR_TYPES = {
    "tennis+heels": {
        "cost": COST["tennis"] + COST["heels"],
        "pairs": SHOE_PAIRS["tennis"] + SHOE_PAIRS["heels"],
        "utility": UTILITY["tennis"] + UTILITY["heels"],
        "boxes": {"tennis": 1, "boot": 0, "heels": 1, "sandals": 0},
    },
    "tennis+sandals": {
        "cost": COST["tennis"] + COST["sandals"],
        "pairs": SHOE_PAIRS["tennis"] + SHOE_PAIRS["sandals"],
        "utility": UTILITY["tennis"] + UTILITY["sandals"],
        "boxes": {"tennis": 1, "boot": 0, "heels": 0, "sandals": 1},
    },
    "boot+heels": {
        "cost": COST["boot"] + COST["heels"],
        "pairs": SHOE_PAIRS["boot"] + SHOE_PAIRS["heels"],
        "utility": UTILITY["boot"] + UTILITY["heels"],
        "boxes": {"tennis": 0, "boot": 1, "heels": 1, "sandals": 0},
    },
    "boot+sandals": {
        "cost": COST["boot"] + COST["sandals"],
        "pairs": SHOE_PAIRS["boot"] + SHOE_PAIRS["sandals"],
        "utility": UTILITY["boot"] + UTILITY["sandals"],
        "boxes": {"tennis": 0, "boot": 1, "heels": 0, "sandals": 1},
    },
}

PAIR_NAMES = list(PAIR_TYPES)
MAX_PAIRED_BOXES = BUDGET // min(data["cost"] for data in PAIR_TYPES.values())
BEST_PAIR_UTILITY = max(data["utility"] for data in PAIR_TYPES.values())
BEST_PAIR_DENSITY = max(data["utility"] / data["cost"] for data in PAIR_TYPES.values())

EXPECTED = {
    "boxes": {"tennis": 15, "boot": 0, "heels": 0, "sandals": 15},
    "pair_types": {"tennis+heels": 0, "tennis+sandals": 15, "boot+heels": 0, "boot+sandals": 0},
    "total_boxes": 30,
    "total_pairs": 1050,
    "total_cost": 49875,
    "total_utility": 86250,
}


def build_solution(pair_counts):
    box_counts = {name: 0 for name in BOXES}
    total_cost = 0
    total_pairs = 0
    total_utility = 0
    total_paired_boxes = 0
    for pair_name, amount in pair_counts.items():
        data = PAIR_TYPES[pair_name]
        total_cost += amount * data["cost"]
        total_pairs += amount * data["pairs"]
        total_utility += amount * data["utility"]
        total_paired_boxes += amount
        for box_name, increment in data["boxes"].items():
            box_counts[box_name] += amount * increment
    total_boxes = sum(box_counts.values())
    if total_cost > BUDGET:
        return None
    if total_boxes < MIN_TOTAL_BOXES:
        return None
    if total_pairs < MIN_SHOE_PAIRS:
        return None
    return {
        "boxes": box_counts,
        "pair_types": dict(pair_counts),
        "total_boxes": total_boxes,
        "total_pairs": total_pairs,
        "total_cost": total_cost,
        "total_utility": total_utility,
    }


@timed
def solve_naive():
    best = None
    for y1 in range(MAX_PAIRED_BOXES + 1):
        for y2 in range(MAX_PAIRED_BOXES + 1 - y1):
            for y3 in range(MAX_PAIRED_BOXES + 1 - y1 - y2):
                for y4 in range(MAX_PAIRED_BOXES + 1 - y1 - y2 - y3):
                    candidate = build_solution(
                        {
                            "tennis+heels": y1,
                            "tennis+sandals": y2,
                            "boot+heels": y3,
                            "boot+sandals": y4,
                        }
                    )
                    best = choose_better(best, candidate)
    return best


@timed
def solve_backtracking():
    best = None

    def backtrack(index, current_counts, current_cost, current_utility, current_pair_boxes):
        nonlocal best
        remaining_budget = BUDGET - current_cost
        remaining_pair_boxes = MAX_PAIRED_BOXES - current_pair_boxes
        optimistic = current_utility + min(remaining_budget * BEST_PAIR_DENSITY, remaining_pair_boxes * BEST_PAIR_UTILITY)
        if best is not None and optimistic <= best["total_utility"]:
            return
        if index == len(PAIR_NAMES):
            candidate = build_solution(current_counts)
            best = choose_better(best, candidate)
            return
        pair_name = PAIR_NAMES[index]
        pair_data = PAIR_TYPES[pair_name]
        max_count = remaining_budget // pair_data["cost"]
        for amount in range(max_count + 1):
            current_counts[pair_name] = amount
            backtrack(
                index + 1,
                current_counts,
                current_cost + amount * pair_data["cost"],
                current_utility + amount * pair_data["utility"],
                current_pair_boxes + amount,
            )
        current_counts[pair_name] = 0

    backtrack(0, {name: 0 for name in PAIR_NAMES}, 0, 0, 0)
    return best


@timed
def solve_reduced_search():
    best = None
    max_tennis_sandals = BUDGET // PAIR_TYPES["tennis+sandals"]["cost"]
    for amount in range(max_tennis_sandals + 1):
        candidate = build_solution(
            {
                "tennis+heels": 0,
                "tennis+sandals": amount,
                "boot+heels": 0,
                "boot+sandals": 0,
            }
        )
        best = choose_better(best, candidate)
    return best


@timed
def solve_dynamic_programming():
    @lru_cache(maxsize=None)
    def dp(index, budget_left, paired_boxes):
        if index == len(PAIR_NAMES):
            total_boxes = 2 * paired_boxes
            if total_boxes < MIN_TOTAL_BOXES:
                return None
            return {"pair_types": {}, "total_utility": 0, "total_pairs": 0}

        pair_name = PAIR_NAMES[index]
        pair_data = PAIR_TYPES[pair_name]
        best_local = None
        max_count = budget_left // pair_data["cost"]
        for amount in range(max_count + 1):
            tail = dp(index + 1, budget_left - amount * pair_data["cost"], paired_boxes + amount)
            if tail is None:
                continue
            candidate = {
                "pair_types": dict(tail["pair_types"], **{pair_name: amount}),
                "total_utility": tail["total_utility"] + amount * pair_data["utility"],
                "total_pairs": tail["total_pairs"] + amount * pair_data["pairs"],
            }
            if candidate["total_pairs"] < 0:
                continue
            if best_local is None or (candidate["total_utility"], candidate["pair_types"]) > (best_local["total_utility"], best_local["pair_types"]):
                best_local = candidate
        return best_local

    compressed = dp(0, BUDGET, 0)
    return build_solution(compressed["pair_types"])


@timed
def solve_branch_and_bound():
    incumbent = build_solution({"tennis+heels": 0, "tennis+sandals": 15, "boot+heels": 0, "boot+sandals": 0})
    best = incumbent

    def branch(index, current_counts, current_cost, current_utility, current_pair_boxes):
        nonlocal best
        remaining_budget = BUDGET - current_cost
        remaining_pair_boxes = MAX_PAIRED_BOXES - current_pair_boxes
        optimistic = current_utility + min(remaining_budget * BEST_PAIR_DENSITY, remaining_pair_boxes * BEST_PAIR_UTILITY)
        if optimistic <= best["total_utility"]:
            return
        if index == len(PAIR_NAMES):
            candidate = build_solution(current_counts)
            best = choose_better(best, candidate)
            return
        pair_name = PAIR_NAMES[index]
        pair_data = PAIR_TYPES[pair_name]
        max_count = remaining_budget // pair_data["cost"]
        order = sorted(
            range(max_count + 1),
            key=lambda amount: (
                amount * pair_data["utility"],
                -amount * pair_data["cost"],
            ),
            reverse=True,
        )
        for amount in order:
            current_counts[pair_name] = amount
            branch(
                index + 1,
                current_counts,
                current_cost + amount * pair_data["cost"],
                current_utility + amount * pair_data["utility"],
                current_pair_boxes + amount,
            )
        current_counts[pair_name] = 0

    branch(0, {name: 0 for name in PAIR_NAMES}, 0, 0, 0)
    return best


## 1. Naive Enumeration


In [ ]:
naive_result, naive_time = solve_naive()
print("Naive result:", naive_result)
print(f"Elapsed time: {naive_time:.6f} seconds")
assert naive_result == EXPECTED


## 2. Backtracking With Pruning


In [ ]:
backtracking_result, backtracking_time = solve_backtracking()
print("Backtracking result:", backtracking_result)
print(f"Elapsed time: {backtracking_time:.6f} seconds")
assert backtracking_result == EXPECTED


## 3. Constraint-Driven Reduced Search


In [ ]:
print(
    "Exact simplification: once the supplier-balance constraint is rewritten in paired boxes, "
    "'tennis+sandals' dominates every other paired purchase for this printed model."
)
reduced_result, reduced_time = solve_reduced_search()
print("Reduced-search result:", reduced_result)
print(f"Elapsed time: {reduced_time:.6f} seconds")
assert reduced_result == EXPECTED


## 4. Dynamic Programming


In [ ]:
dp_result, dp_time = solve_dynamic_programming()
print("Dynamic-programming result:", dp_result)
print(f"Elapsed time: {dp_time:.6f} seconds")
assert dp_result == EXPECTED


## 5. Branch And Bound


In [ ]:
bnb_result, bnb_time = solve_branch_and_bound()
print("Branch-and-bound result:", bnb_result)
print(f"Elapsed time: {bnb_time:.6f} seconds")
assert bnb_result == EXPECTED


## Summary


In [ ]:
methods = [
    ("Naive enumeration", naive_result, naive_time),
    ("Backtracking with pruning", backtracking_result, backtracking_time),
    ("Constraint-driven reduced search", reduced_result, reduced_time),
    ("Dynamic programming", dp_result, dp_time),
    ("Branch and Bound", bnb_result, bnb_time),
]

for method_name, result, elapsed in methods:
    print(f"{method_name:35s} utility={result['total_utility']:6d} cost={result['total_cost']:6d} time={elapsed:.6f}s")

assert all(result == EXPECTED for _, result, _ in methods)
print("\nAll five exact methods reproduce the published solution for the printed model.")
